In [18]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import torch 
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid

from dataset_subset import make_resnet_demo_subsets

In [19]:
SEED = 42
torch.manual_seed(SEED)

In [20]:
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

In [21]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

resnet_train_full = datasets.CIFAR10(
    root=Path("data"),
    train=True,
    download=True,
    transform=resnet_transform,
)
resnet_test_full = datasets.CIFAR10(
    root=Path("data"),
    train=False,
    download=True,
    transform=resnet_transform,
)

resnet_train_dataset = resnet_train_full
resnet_test_dataset = resnet_test_full

USE_SMALL_SUBSET = True

if USE_SMALL_SUBSET:
    resnet_train_dataset, resnet_test_dataset = make_resnet_demo_subsets(
        resnet_train_full,
        resnet_test_full,
        seed=SEED,
    )

resnet_train_loader = DataLoader(
    resnet_train_dataset,
    batch_size=32,
    shuffle=True,
)
resnet_test_loader = DataLoader(
    resnet_test_dataset,
    batch_size=32,
    shuffle=False,
)

print(f"ResNet train images: {len(resnet_train_dataset)}")
print(f"ResNet test images: {len(resnet_test_dataset)}")

ResNet train images: 200
ResNet test images: 100


In [22]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False
        
model.fc = nn.Linear(model.fc.in_features, 10)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam((p for p in model.parameters() if p.requires_grad), lr=0.001)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("In features: ", model.fc.in_features)
print("Trainable params: ", trainable_params)
print("Total params: ", total_params)

In features:  512
Trainable params:  5130
Total params:  11181642


In [23]:
def train_one_epoch(model, dataloader, loss_fn, optimizer):
    model.train()
    
    total_loss = 0.0
    totas_items = 0
    total_correct = 0
    
    for images, labels in dataloader:
        # обнуляємо градієнти, щоб модель правильно началась
        optimizer.zero_grad(set_to_none=True)
        
        # forward pass - проходимо вперед
        logits = model(images)
    
        # рахуємо помилку
        loss = loss_fn(logits, labels)
        
        # backpropagation - рахуємо градієнти
        loss.backward()
        
        # оновлює ваги моделі, використовує пораховані градієнти
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        totas_items += len(labels)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        
    return {
        "loss": total_loss / totas_items,
        "accuracy": total_correct / totas_items
    }

In [24]:
@torch.no_grad()
def evaluate(model, dataloader, loss_fn):
    model.eval()
    
    total_loss = 0.0
    total_items = 0
    total_correct = 0
    
    for images, labels in dataloader:
        # forward pass - проходимо вперед
        logits = model(images)
        
        # рахуємо помилку
        loss = loss_fn(logits, labels)
        
        total_loss += loss.item() * labels.size(0)
        total_items += labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        
    return {
        "loss": total_loss / total_items,
        "accuracy": total_correct / total_items
    }

In [25]:
def model_fit(model, train_dataloader, test_dataloader, loss_fn, optimizer, epochs=10):
    history = []    
    
    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(model, train_dataloader, loss_fn, optimizer)
        test_metrics = evaluate(model, test_dataloader, loss_fn)
    
        row = {
            "epoch": epoch,
            "train_loss": train_metrics['loss'],
            "train_accuracy": train_metrics['accuracy'],
            "test_loss": test_metrics['loss'],
            "test_accuracy": test_metrics['accuracy']
        }
        history.append(row)
        print(row)
        
    return pd.DataFrame(history)

In [26]:
history = model_fit(model, resnet_train_loader, resnet_test_loader, loss_fn, optimizer)
history

{'epoch': 1, 'train_loss': 2.3619533252716063, 'train_accuracy': 0.125, 'test_loss': 2.3627928733825683, 'test_accuracy': 0.18}
{'epoch': 2, 'train_loss': 2.0448962211608888, 'train_accuracy': 0.285, 'test_loss': 2.1193777179718016, 'test_accuracy': 0.22}
{'epoch': 3, 'train_loss': 1.7742192840576172, 'train_accuracy': 0.515, 'test_loss': 1.9239680004119872, 'test_accuracy': 0.32}
{'epoch': 4, 'train_loss': 1.6027005767822267, 'train_accuracy': 0.525, 'test_loss': 1.7999829292297362, 'test_accuracy': 0.41}
{'epoch': 5, 'train_loss': 1.3884369659423828, 'train_accuracy': 0.745, 'test_loss': 1.695894594192505, 'test_accuracy': 0.45}
{'epoch': 6, 'train_loss': 1.2827234363555908, 'train_accuracy': 0.745, 'test_loss': 1.5699835395812989, 'test_accuracy': 0.55}
{'epoch': 7, 'train_loss': 1.1730335712432862, 'train_accuracy': 0.775, 'test_loss': 1.4892247104644776, 'test_accuracy': 0.51}
{'epoch': 8, 'train_loss': 1.0376081037521363, 'train_accuracy': 0.835, 'test_loss': 1.427231879234314, '

,epoch,train_loss,train_accuracy,test_loss,test_accuracy
0,1,2.361953,0.125,2.362793,0.18
1,2,2.044896,0.285,2.119378,0.22
2,3,1.774219,0.515,1.923968,0.32
3,4,1.602701,0.525,1.799983,0.41
4,5,1.388437,0.745,1.695895,0.45
5,6,1.282723,0.745,1.569984,0.55
6,7,1.173034,0.775,1.489225,0.51
7,8,1.037608,0.835,1.427232,0.51
8,9,0.957663,0.840,1.375736,0.60
9,10,0.860211,0.865,1.311014,0.61
